In [1]:
import numpy as np
from tqdm.auto import tqdm

import config
from dataset import EEGDataPipeline
from snn import SNN

In [2]:
def train_stdp(model, train_loader, epochs=config.epochs):
    history = {'spikes': [], 'weight_changes': []}
    for epoch in range(epochs):
        W_before = model.W.copy()
        total_spikes = 0
        total_windows = 0
        with tqdm(total=len(train_loader.dataset), desc=f"epoch {epoch + 1}/{epochs}", unit="window") as progress:
            for signals, _ in train_loader:
                signals = signals.detach().cpu().numpy()
                for eeg in signals:
                    _, output_spikes, _ = model.forward(eeg, learn=True)
                    total_spikes += output_spikes.sum()
                    total_windows += 1
                    progress.update()
        weight_change = np.linalg.norm(model.W - W_before)
        mean_spikes = total_spikes / total_windows
        history['spikes'].append(float(mean_spikes))
        history['weight_changes'].append(float(weight_change))
        print(f'epoch: {epoch + 1}/{epochs}; mean spikes: {mean_spikes:.2f}; weight change: {weight_change:.6f}')
    return history

In [ ]:
pipeline = EEGDataPipeline(channel_names=("O1", "OZ", "O2"))
pipeline.split_data()
train_loader, val_dataloader, _ = pipeline.get_dataloaders()
model = SNN()
W_initial = model.W.copy()
history = train_stdp(model=model, train_loader=train_loader)
W_trained = model.W.copy()

processing eeg:   0%|          | 0/76 [00:00<?, ?subject/s]

processing eeg:   0%|          | 0/16 [00:00<?, ?subject/s]

processing eeg:   0%|          | 0/17 [00:00<?, ?subject/s]

epoch 1/3:   0%|          | 0/2280 [00:00<?, ?window/s]

epoch: 1/3; mean spikes: 216.24; weight change: 1.963754


epoch 2/3:   0%|          | 0/2280 [00:00<?, ?window/s]

epoch: 2/3; mean spikes: 223.22; weight change: 0.039988


epoch 3/3:   0%|          | 0/2280 [00:00<?, ?window/s]

epoch: 3/3; mean spikes: 223.22; weight change: 0.041487
